# Inferência (ModelLoader + Predictor)

Treina um modelo minúsculo, salva, recarrega pelo `ModelLoader` e prevê
com o `Predictor` — o caminho real de detecção, incluindo a leitura do
`input_contract` (temperatura/EER) e o fallback ONNX→TF.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

_REPO_URL = "https://github.com/thierrybraga/XFakeSong.git"


def _project_root() -> Path:
    p = Path.cwd()
    while not (p / "app").exists() and p.parent != p:
        p = p.parent
    return p


# Colab/Kaggle limpos: clona o repositório se o projeto não estiver no disco.
if not (_project_root() / "app").exists():
    if not Path("XFakeSong").exists():
        print("Clonando XFakeSong...")
        subprocess.run(["git", "clone", "--depth", "1", _REPO_URL], check=True)
    os.chdir("XFakeSong")

_DEPS = {"librosa": "librosa>=0.10", "soundfile": "soundfile>=0.12",
         "pywt": "PyWavelets>=1.4"}
_missing = [s for m, s in _DEPS.items() if importlib.util.find_spec(m) is None]
if _missing:
    print("Instalando dependências de áudio:", *_missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing],
                   check=True)

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT, "| Colab:", "google.colab" in sys.modules)

## Reprodutibilidade (semente numpy + TensorFlow)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(0)
except Exception:  # pragma: no cover
    np.random.seed(0)

## 1. Treinar + salvar um modelo de demonstração

In [ ]:
import tempfile
import numpy as np
from app.core.interfaces.base import ProcessingStatus
from app.domain.services.training_service import TrainingService

rng = np.random.default_rng(1)
n, T, F = 240, 32, 16
X = rng.standard_normal((n, T, F)).astype("float32")
y = rng.integers(0, 2, n).astype("int64")
X[y == 1] += 0.6

workdir = Path(tempfile.mkdtemp(prefix="nb_infer_"))
npz = workdir / "ds.npz"
np.savez(npz, X_train=X, y_train=y)
models_dir = workdir / "models"
res = TrainingService(models_dir=str(models_dir)).train_model(
    architecture="MultiscaleCNN", dataset_path=str(npz),
    config={"epochs": 2, "batch_size": 32, "model_name": "nb_infer"},
)
assert res.status == ProcessingStatus.SUCCESS, res.errors
print("treinado e salvo em", models_dir)

## 2. Carregar e prever

In [ ]:
from app.domain.services.detection.model_loader import ModelLoader
from app.domain.services.detection.predictor import Predictor

loader = ModelLoader(models_dir=str(models_dir))
loader.load_available_models()
mi = loader.get_model("nb_infer")
print("modelo:", mi.architecture, "| input_shape:", mi.input_shape)
print("temperatura calibrada:", mi.temperature,
      "| eer_threshold:", mi.eer_threshold)

sample = X[:1][0]  # uma amostra (T, F)
out = Predictor().predict(mi, sample)
print("resultado:", out.data)

## Notas

- `ModelLoader` lê o `input_contract` do `_config.json` (temperatura/EER/OOD)
  e o `Predictor` os aplica automaticamente.
- Se houver um `nb_infer.onnx` ao lado e `onnxruntime` instalado, a
  inferência usa ONNX (mais rápida em CPU) com fallback para o TF.

## 3. Inferência pelo PIPELINE DO SISTEMA (`DetectionService`)

`ModelLoader` + `Predictor` acima são os internos. O
`DetectionService.detect_single` é o **mesmo caminho de alto nível que a API
e o app Gradio usam**: recebe **áudio bruto**, extrai as features pelo
`input_contract` e prevê. É a prova de que um modelo treinado nos notebooks
roda, sem adaptação, no sistema principal.

In [ ]:
from app.core.interfaces.audio import AudioData
from app.domain.services.detection_service import DetectionService

# Mesma instância que o app cria — só muda o models_dir (aponta p/ o do
# notebook). No app, o default já é "app/models" (Gradio E API).
ds = DetectionService(models_dir=str(models_dir), create_default_models=False)

# Entrada REAL do pipeline: áudio bruto de 1 s (16 kHz).
t = np.linspace(0, 1.0, 16000, endpoint=False)
audio = AudioData(
    samples=(0.6 * np.sin(2 * np.pi * 220 * t)).astype("float32"),
    sample_rate=16000, duration=1.0,
)

out = ds.detect_single(audio, model_name="nb_infer")
print("status :", out.status.value)
print("is_fake:", out.data.is_fake,
      "| confiança:", round(float(out.data.confidence), 4))

## Onde salvar para o app principal carregar o seu modelo

O par **`<nome>.keras` + `<nome>_config.json`** é auto-suficiente: o
`DetectionService` varre o `models_dir` e expõe o modelo por
`model_name="<nome>"`. Para o seu modelo aparecer no app, copie os dois
arquivos para o diretório que cada interface lê:

| Interface | Diretório lido | Como subir o modelo |
| --- | --- | --- |
| **Gradio** (`python main.py --gradio`) | `app/models/` | copie `<nome>.keras` + `<nome>_config.json` para `app/models/` |
| **API FastAPI** | `app/models/` | idem — mesmo diretório |

> Dica: `app/models/` é o **default** do `TrainingService` e do
> `DetectionService`, então treinar sem passar `models_dir` já põe o modelo
> no lugar certo para as duas interfaces. Sem o `_config.json` ao lado, o
> serviço cai no contrato do registry (ainda funciona, mas sem a
> temperatura/EER calibrados do seu treino).